## Overview

Prompt engineering is the art of constructing inputs to language models that reliably produce
the desired output. This tutorial catalogs the most important prompt patterns used in production
systems, implements each as a Python function, and explains when — and why — each pattern works.

No API key is required. All examples demonstrate prompt *construction* and can be fed to any
LLM backend (OpenAI, Anthropic, HuggingFace `pipeline`, Ollama, etc.).

## Helper: Prompt Renderer


In [ ]:
#| eval: false
def show_prompt(prompt: str, label: str = "Prompt") -> None:
    """Print a prompt with a visible boundary."""
    bar = "─" * 70
    print(f"\n┌─ {label} {'─' * max(0, 66 - len(label))}┐")
    for line in prompt.splitlines():
        print(f"│ {line}")
    print(f"└{bar}┘\n")


## Pattern 1: Zero-Shot

Give the model a task description and input — no examples.


In [ ]:
#| eval: false
def zero_shot(task: str, input_text: str) -> str:
    return f"{task}\n\nInput: {input_text}\nOutput:"


task = "Classify the sentiment of the following financial news headline as POSITIVE, NEGATIVE, or NEUTRAL."
headline = "Federal Reserve signals three rate cuts in 2025 amid easing inflation."

show_prompt(zero_shot(task, headline), label="Zero-Shot")


**When to use**: The task is straightforward and well-defined. Zero-shot minimises prompt length,
which matters for rate-limited APIs.

## Pattern 2: Few-Shot

Include labelled examples (shots) to demonstrate the desired format and reasoning style.


In [ ]:
#| eval: false
def few_shot(task: str, examples: list[dict], input_text: str,
             input_key: str = "input", output_key: str = "output") -> str:
    prompt = task + "\n\n"
    for ex in examples:
        prompt += f"{input_key.capitalize()}: {ex[input_key]}\n"
        prompt += f"{output_key.capitalize()}: {ex[output_key]}\n\n"
    prompt += f"{input_key.capitalize()}: {input_text}\n{output_key.capitalize()}:"
    return prompt


shots = [
    {"input": "Apple surpasses $3T market cap for first time.",  "output": "POSITIVE"},
    {"input": "Tech layoffs hit record high in Q1.",             "output": "NEGATIVE"},
    {"input": "Quarterly earnings report released as scheduled.","output": "NEUTRAL"},
]

show_prompt(few_shot(task, shots, headline), label="Few-Shot (k=3)")


**When to use**: The task requires a specific output format, label set, or reasoning style that
the model may not follow reliably in zero-shot mode.

## Pattern 3: Chain-of-Thought (CoT)

Ask the model to reason step-by-step before giving the final answer.


In [ ]:
#| eval: false
def chain_of_thought(task: str, examples: list[dict], question: str) -> str:
    """Each example must have: question, reasoning, answer."""
    prompt = task + "\n\n"
    for ex in examples:
        prompt += f"Question: {ex['question']}\n"
        prompt += f"Reasoning: {ex['reasoning']}\n"
        prompt += f"Answer: {ex['answer']}\n\n"
    prompt += f"Question: {question}\nReasoning:"
    return prompt


cot_task = (
    "You are a financial analyst. "
    "Show your reasoning step by step before giving the final answer."
)
cot_examples = [
    {
        "question": "A company's revenue grew from $400M to $500M. What is the YoY growth rate?",
        "reasoning": "Growth = (500 - 400) / 400 = 100 / 400 = 0.25.",
        "answer":    "25% year-over-year revenue growth.",
    },
    {
        "question": "Operating income is $60M on $400M revenue. What is the operating margin?",
        "reasoning": "Operating margin = operating income / revenue = 60 / 400 = 0.15.",
        "answer":    "15% operating margin.",
    },
]
question = "A firm's net income fell from $120M to $96M. By what percentage did it decline?"

show_prompt(chain_of_thought(cot_task, cot_examples, question), label="Chain-of-Thought")


**When to use**: Multi-step arithmetic, logical inference, or any task where intermediate steps
significantly improve final answer accuracy.

## Pattern 4: Role Prompting

Assign a persona and context to prime the model's response style and domain knowledge.


In [ ]:
#| eval: false
def role_prompt(role: str, context: str, task: str) -> str:
    return (
        f"You are {role}.\n\n"
        f"Context:\n{context}\n\n"
        f"Task: {task}"
    )


role    = "a senior equity research analyst at a bulge-bracket investment bank"
context = """
Excerpt from Item 1A (Risk Factors) of a Form 10-K filing:
'The Company is subject to intense competition from established players and new entrants.
Rapid technological innovation may render existing products obsolete. Additionally,
the Company relies heavily on a small number of key customers for a significant
portion of its revenue.'
"""
task    = ("Summarise the top three business risks in bullet points, "
           "rated High / Medium / Low severity, in under 100 words.")

show_prompt(role_prompt(role, context, task), label="Role Prompting")


## Pattern 5: Structured Output (JSON)

Instruct the model to respond strictly in a parseable format.


In [ ]:
#| eval: false
import json, textwrap

def structured_json_prompt(role: str, context: str, schema: dict, task: str) -> str:
    schema_str = json.dumps(schema, indent=2)
    return (
        f"You are {role}.\n\n"
        f"Context:\n{context}\n\n"
        f"Task: {task}\n\n"
        "Respond ONLY with valid JSON that matches this schema — no prose, no markdown fences:\n"
        f"{schema_str}"
    )


schema = {
    "risks": [
        {
            "rank":        "integer (1 = highest)",
            "category":    "string",
            "description": "string (one sentence)",
            "severity":    "High | Medium | Low"
        }
    ]
}

prompt = structured_json_prompt(
    role=role,
    context=context,
    schema=schema,
    task="Extract the top 3 risk categories from the context.",
)
show_prompt(prompt, label="Structured JSON Output")


**When to use**: Downstream code needs to parse the response (e.g., insert into a database,
call another API, render in a UI).

## Pattern 6: System Prompts

System prompts (supported by OpenAI, Anthropic, and most chat-tuned models) set persistent
instructions that frame all subsequent turns.


In [ ]:
#| eval: false
def build_messages(system: str, user: str) -> list[dict]:
    """Constructs an OpenAI-compatible messages list."""
    return [
        {"role": "system",  "content": system},
        {"role": "user",    "content": user},
    ]


system = (
    "You are a financial assistant for Boston University AD698. "
    "Always ground your answers in the provided context. "
    "If uncertain, say so explicitly. "
    "Format responses as numbered lists when listing multiple items."
)
user = "What are the key differences between BERT and GPT-style language models?"

msgs = build_messages(system, user)
for m in msgs:
    show_prompt(m["content"], label=f"{m['role'].upper()} message")


## Pattern 7: Prompt Chaining

Break a complex task into a sequence of simpler prompts, passing each output as input to the next.


In [ ]:
#| eval: false
def chain_prompts(steps: list[dict], initial_input: str) -> list[dict]:
    """
    steps: list of dicts with 'name' and 'template' (use {input} placeholder).
    Returns list of {step, prompt, output_placeholder} for each step.
    """
    history = []
    current = initial_input
    for step in steps:
        prompt = step["template"].format(input=current)
        history.append({"step": step["name"], "prompt": prompt})
        # In production, you would call your LLM here:
        # current = call_llm(prompt)
        # For demo, we just pass the prompt forward as the simulated output
        current = f"[output of step: {step['name']}]"
    return history


pipeline_steps = [
    {
        "name": "Extract Key Facts",
        "template": (
            "Extract the 5 most important financial facts from the following 10-K excerpt.\n"
            "Return one fact per line.\n\nExcerpt:\n{input}"
        ),
    },
    {
        "name": "Assess Risk Level",
        "template": (
            "Given the following financial facts, rate the overall investment risk as "
            "Low / Medium / High and provide a one-sentence justification.\n\nFacts:\n{input}"
        ),
    },
    {
        "name": "Generate Recommendation",
        "template": (
            "Based on the following risk assessment, write a two-sentence investment "
            "recommendation suitable for a retail investor.\n\nRisk Assessment:\n{input}"
        ),
    },
]

filing_excerpt = """
Revenue grew 18% YoY to $12.4B driven by cloud segment.
Net income declined 5% due to elevated R&D spend.
The company faces antitrust scrutiny in three jurisdictions.
Customer concentration risk: top 3 customers = 42% of revenue.
"""

chain_log = chain_prompts(pipeline_steps, filing_excerpt)
for step in chain_log:
    show_prompt(step["prompt"], label=step["step"])


## Pattern 8: Negative Examples

Tell the model explicitly what NOT to do — reduces hallucination and format violations.


In [ ]:
#| eval: false
def negative_example_prompt(task: str, do: list[str], dont: list[str],
                             input_text: str) -> str:
    do_str   = "\n".join(f"  ✓ {item}" for item in do)
    dont_str = "\n".join(f"  ✗ {item}" for item in dont)
    return (
        f"{task}\n\n"
        f"DO:\n{do_str}\n\n"
        f"DO NOT:\n{dont_str}\n\n"
        f"Input:\n{input_text}\n\nOutput:"
    )


prompt = negative_example_prompt(
    task="Summarise the following 10-K risk factor excerpt.",
    do=[
        "Use bullet points",
        "Limit to 3 bullets",
        "Cite the specific risk (e.g., 'Regulatory risk')",
    ],
    dont=[
        "Add information not present in the excerpt",
        "Include your personal opinion",
        "Use more than 15 words per bullet",
    ],
    input_text=filing_excerpt,
)
show_prompt(prompt, label="Negative Examples")


## Exercises

1. **Zero-Shot vs Few-Shot Gap**: For the sentiment task, record the accuracy of a zero-shot prompt vs a 5-shot prompt against 20 hand-labelled headlines. Calculate the accuracy improvement.

2. **CoT Ablation**: On the financial arithmetic examples, compare accuracy with and without chain-of-thought reasoning. Use 10 questions with known answers.

3. **JSON Schema Validation**: Call your favourite LLM API with the structured JSON prompt. Parse the response with `json.loads()`. What percentage of 20 responses parse successfully without errors?

4. **Chain Evaluation**: Implement the prompt chain with an actual LLM API call. Evaluate whether the final recommendation for 5 filing excerpts is consistent with the risk level determined in step 2.

5. **System Prompt Engineering**: Experiment with three different system prompts (formal analyst, simple explainer, risk-focused) for the same user question. How does the tone and structure of the response change?
